In [ ]:
import albumentations as A
import numpy as np

# Define aggressive augmentations
train_transforms = A.Compose([
    A.Rotate(limit=5, p=0.5),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.5),
    A.OneOf([
        A.Blur(blur_limit=3, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
    ], p=0.5),
    # This is the secret sauce for handwriting:
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.5),
    ], p=0.5),
])

In [ ]:
image_processor = ViTImageProcessor.from_pretrained(ModelConfig.MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained("aisingapore/sealion-bert-base", use_fast=False, trust_remote_code=True)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-stage1")

model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 2
model.config.length_penalty = 1
model.config.num_beams = 4
model.config.dropout = 0.2

In [ ]:
training_args = Seq2SeqTrainingArguments(
        output_dir="./trocr-mmnrc-outputs",
        predict_with_generate=True,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        
        # Architecture & Hardware Constraints (Single NVIDIA L4 setup)
        fp16=True,                           # Mixed-precision training
        gradient_checkpointing=True,         # Memory optimization
        per_device_train_batch_size=4,       # Batch size
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,       # Accumulation step
        
        # Optimization & Scheduling
        num_train_epochs=30,                 # Total epochs
        learning_rate=1e-5,                  # LR
        weight_decay=0.01,                   # Weight decay
        lr_scheduler_type="cosine",          # Cosine schedule
        warmup_ratio=0.1,                    # 10% warmup
        
        # Privacy / Logging sanitization
        report_to="tensorboard",          
    )

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

trainer.train()
trainer.save_model("./trocr-mmnrc-final")